# 1. Chargement et préparation des données

In [1]:
import pandas as pd
import numpy as np
import plotly.express as px

# Test statistiques
from scipy.stats import ttest_ind, chi2_contingency
from scipy import stats
from scipy.stats import pearsonr
from scipy.stats import ttest_ind
import statsmodels.formula.api as smf

In [2]:
# Chargement du dataset
df = pd.read_csv("Speed+Dating+Data.csv", encoding="latin1")

# Sélection des variables utiles
df_o = df[[
    "iid","pid","gender","match","dec",
    "attr","fun","intel","sinc","amb","shar",
    "attr_o","fun_o","intel_o","sinc_o","shar_o",
    "int_corr","samerace","age","age_o"
]]

# Nettoyage simple
df = df.dropna(subset=["attr","fun","intel","sinc","shar"])

In [3]:
df.head()

,iid,id,gender,idg,condtn,wave,round,position,positin1,order,...,attr3_3,sinc3_3,intel3_3,fun3_3,amb3_3,attr5_3,sinc5_3,intel5_3,fun5_3,amb5_3
0,1,1.0,0,1,1,1,10,7,NaN,4,...,5.0,7.0,7.0,7.0,7.0,NaN,NaN,NaN,NaN,NaN
1,1,1.0,0,1,1,1,10,7,NaN,3,...,5.0,7.0,7.0,7.0,7.0,NaN,NaN,NaN,NaN,NaN
2,1,1.0,0,1,1,1,10,7,NaN,10,...,5.0,7.0,7.0,7.0,7.0,NaN,NaN,NaN,NaN,NaN
3,1,1.0,0,1,1,1,10,7,NaN,5,...,5.0,7.0,7.0,7.0,7.0,NaN,NaN,NaN,NaN,NaN
4,1,1.0,0,1,1,1,10,7,NaN,7,...,5.0,7.0,7.0,7.0,7.0,NaN,NaN,NaN,NaN,NaN


In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 7206 entries, 0 to 8377
Columns: 195 entries, iid to amb5_3
dtypes: float64(174), int64(13), object(8)
memory usage: 10.8+ MB


In [5]:
df.describe()

,iid,id,gender,idg,condtn,wave,round,position,positin1,order,...,attr3_3,sinc3_3,intel3_3,fun3_3,amb3_3,attr5_3,sinc5_3,intel5_3,fun5_3,amb5_3
count,7206.000000,7205.000000,7206.000000,7206.000000,7206.000000,7206.000000,7206.000000,7206.000000,5607.000000,7206.000000,...,3394.000000,3394.000000,3394.000000,3394.000000,3394.000000,1676.000000,1676.000000,1676.000000,1676.000000,1676.000000
mean,284.524008,9.009577,0.507771,17.435332,1.821954,11.379406,16.796003,9.056342,9.348850,8.805162,...,7.289629,8.139953,8.406600,7.710371,7.489982,6.802506,7.672434,7.972554,7.176611,7.146778
std,159.484102,5.515919,0.499974,10.998261,0.382578,6.031294,4.420131,5.514213,5.645264,5.429460,...,1.588187,1.606108,1.470096,1.728414,1.977933,1.486181,1.513159,1.346469,1.599202,1.706510
min,1.000000,1.000000,0.000000,1.000000,1.000000,1.000000,5.000000,1.000000,1.000000,1.000000,...,2.000000,2.000000,3.000000,2.000000,1.000000,2.000000,2.000000,4.000000,1.000000,1.000000
25%,155.000000,4.000000,0.000000,8.000000,2.000000,7.000000,14.000000,4.000000,4.000000,4.000000,...,6.250000,7.000000,8.000000,7.000000,7.000000,6.000000,7.000000,7.000000,6.000000,6.000000
50%,280.000000,8.000000,1.000000,16.000000,2.000000,11.000000,18.000000,8.000000,9.000000,8.000000,...,7.000000,8.000000,8.000000,8.000000,8.000000,7.000000,8.000000,8.000000,7.000000,7.000000
75%,413.000000,13.000000,1.000000,26.000000,2.000000,15.000000,20.000000,13.000000,14.000000,13.000000,...,8.000000,9.000000,9.000000,9.000000,9.000000,8.000000,9.000000,9.000000,8.000000,8.000000
max,552.000000,22.000000,1.000000,44.000000,2.000000,21.000000,22.000000,22.000000,22.000000,22.000000,...,12.000000,12.000000,12.000000,12.000000,12.000000,10.000000,10.000000,10.000000,10.000000,10.000000


In [6]:
df.shape

(7206, 195)

## 1.1 - Transformation

In [7]:
# Ajout d'une colonne "gender_text" pour garder celle d'origine, mais afficher les graphiques avec le genre

df["gender_text"] = df["gender"].map({
    0:"Femme",
    1:"Homme"
})

# Ajout d'une colonne "race_text" pour garder celle d'origine, mais afficher les graphiques avec l'orgine ethnique

df["race_text"] = df["race"].map({
    1:"Black/African American",
    2:"European/Caucasian-American",
    3:"Latino/Hispanic American",
    4:"Asian/Pacific Islander/Asian-American",
    5:"Native American",
    6:"Other"
})

# Ajout d'une colonne "goal_text" pour garder celle d'origine, mais afficher les graphiques avec l'objectif du date

df["goal_text"] = df["goal"].map({
    1:"Seemed like a fun night out",
    2:"To meet new people",
    3:"To get a date",
    4:"Looking for a serious relationship",
    5:"To say I did it",
    6:"Other"
})

## 1.2 Harmonisation des variables

In [8]:
# Harmonisation des blocs concernés, transformation d'une échelle absolue en échelle relative

cols_attr1 = ["attr1_1","sinc1_1","intel1_1","fun1_1","amb1_1","shar1_1"]
cols_attr2 = ["attr2_1","sinc2_1","intel2_1","fun2_1","amb2_1","shar2_1"]
cols_attr4 = ["attr4_1","sinc4_1","intel4_1","fun4_1","amb4_1","shar4_1"]

all_relative_blocks = cols_attr1 + cols_attr2 + cols_attr4

mask = df["wave"].isin([6,7,8,9]) # Sélection des vagues 6 à 9 dans mon masque

for block in [cols_attr1, cols_attr2, cols_attr4]:
    row_sums = df.loc[mask, block].sum(axis=1) # On calcule pour chaque participant, la somme des 6 scores
    df.loc[mask, block] = df.loc[mask, block].div(row_sums.replace(0,np.nan), axis=0) * 100 # on divise chaque valeur par la somme de la ligne, puis x100. On protège la somme si Nan.

# Vérification de la bonne harmonisation

df.groupby("wave")[cols_attr1].mean()

,attr1_1,sinc1_1,intel1_1,fun1_1,amb1_1,shar1_1
wave,,,,,,
1,32.985185,11.020635,20.020106,18.904233,7.316931,9.751852
2,24.226721,18.344130,23.989879,16.463563,8.751012,8.953441
3,21.165644,15.705521,21.717791,19.049080,11.601227,12.392638
4,19.806922,18.444444,21.706740,16.477231,11.038251,12.198543
5,30.000000,15.571429,18.628571,19.515152,10.849057,8.301887
6,16.059925,18.685290,17.714140,17.706820,12.767588,17.066237
7,15.330216,17.553746,19.080570,18.223271,15.916410,13.895787
8,16.556684,18.605061,18.332264,17.374062,15.998127,13.133802
9,16.603764,17.297799,19.218654,17.832682,14.139466,14.907635


Les vagues 6–9 permettaient de donner 10 à TOUTES les catégories.

Donc après normalisation :

Quelqu’un qui mettait 10 partout , devient distribution égale (~16.6% chacun)

# 2. Statistiques descriptives

In [9]:
# Nombre total d'observations
n_dates = len(df)

# Nombre de participants
n_participants = df["iid"].nunique()

# taux de match
match_rate = df["match"].mean()

print("Nombre de speed dates :", n_dates)
print("Nombre de participants :", n_participants)
print("Match rate :", round(match_rate,3))

Nombre de speed dates : 7206
Nombre de participants : 540
Match rate : 0.173


In [10]:
# Nombre d'hommes et de femmes dans chaque vague

total_men_women_by_waves = (df
                            .groupby(["wave","iid"])["gender_text"]
                            .value_counts()
                            .reset_index()
                            .groupby(["wave","gender_text"])["iid"]
                            .count()
                            .reset_index(name="count")
                            )

display(total_men_women_by_waves)

# Visualisation 

fig = px.bar(total_men_women_by_waves, 
             x="wave",
             y="count",
             color="gender_text",
             text_auto=True,
             title="Répartition du nombre de candidats par vague",
             barmode='group',
             color_discrete_map={
                 "Femme":"#E75480",
                 "Homme":"#1f77b4"},
            labels={
                "wave":"Vagues",
                "gender_text":"Genre",
                "count":"Nombre de participants"
            })
fig.show()

,wave,gender_text,count
0,1,Femme,10
1,1,Homme,10
2,2,Femme,19
3,2,Homme,16
4,3,Femme,10
5,3,Homme,10
6,4,Femme,18
7,4,Homme,18
8,5,Femme,9
9,5,Homme,10


In [11]:
# Nombre de participants uniques par âge :

unique_students_by_age = (
    df.groupby(["age","gender_text"])["iid"]
      .nunique()
      .reset_index(name="n_persons")
      .sort_values("age")
)

# Visualisation en barre
fig = px.line(unique_students_by_age, 
             x="age",
             y="n_persons",
             title="Nombre de candidats par âge",
             labels={
                 "n_persons":"nombre de participants",
                 "age":"Age des participants",
                 "gender_text":"Genre"
             },
             color="gender_text",
             #text_auto=True,
             #barmode="group",
             color_discrete_map={
                 "Femme":"#E75480",
                 "Homme":"#1f77b4"})
fig.show()

# visualisation en boxplot
fig = px.box(unique_students_by_age,
             x="gender_text", 
             y="age",
             title="Dispersion des candidats par âges",
             labels={
                 "gender_text":"Genre"
             })
fig.show()

In [12]:
# Ratio de la représentativité par ethnie d'origine

race_count = df.groupby("race_text")["iid"].nunique().reset_index(name="n_persons")
race_count["value_pct"] = round(race_count["n_persons"] / race_count["n_persons"].sum() * 100,2)

display(race_count.sort_values(by="value_pct",ascending=False))

# Visualisation camembert
fig = px.pie(race_count, values="value_pct",names="race_text",title="Représentativité des candidats par ethnie d'origine")
fig.show()

,race_text,n_persons,value_pct
2,European/Caucasian-American,298,55.81
0,Asian/Pacific Islander/Asian-American,132,24.72
3,Latino/Hispanic American,42,7.87
4,Other,36,6.74
1,Black/African American,26,4.87


In [13]:
# Totaux uniques des raisons du speed meeting, et ratio de match

student_level = (
    df.groupby(["iid", "goal_text"])["match"]
      .sum()
      .reset_index()
)

reason_table = (
    student_level.groupby("goal_text")
    .agg(
        n_students=("iid", "count"),
        avg_matches=("match", "mean")
    )
    .reset_index().round(0)
)

reason_table = reason_table.sort_values(by='avg_matches',ascending=False)

# Visualisation : Nombre de participants par objectifs
fig = px.bar(
    reason_table,
    x="goal_text",
    y="n_students",
    title="Nombre de participants par objectif",
    labels={
        "n_students":"Nombre de participants",
        "goal_text":"Objectif"
    }

)
fig.show()

# Visualisation : Nombre moyens de match par objectif
fig = px.bar(
    reason_table,
    x="goal_text",
    y="avg_matches",
    title="Nombre moyen de matchs par objectif de participation",
    labels={
        "avg_matches":"Nombre de matchs moyens",
        "goal_text":"Objectif"
    }

)
fig.show()

In [14]:
# Avoir une table des objectifs de rendez-vous, par genre

table_goal_gender_match = df.groupby(["goal_text", "gender_text"])["match"].sum().round(2).reset_index(name="match").sort_values(by="match",ascending=False)

# Visualiser le taux de match par objectif de rendez-vous et genre
fig = px.bar(table_goal_gender_match, 
             x="goal_text",
             y="match",
             color="gender_text",
             barmode="group",
             color_discrete_map={
                 "Femme":"#E75480",
                 "Homme":"#1f77b4"},
            labels={
                "gender_text":"Genre",
                "goal_text":"Objectif",
                "match":"Nombre de match"
            },
            title="Nombre de matchs par objectif du speed dating, par genre"
            )
fig.show()

**EN RESUME :**

- Une majorité de participants ont répondu vouloir participer pour s'amuser et rencontrer de nouvelles personnes. A l'inverse, très peu de personnes ont répondu être là pour réellement faire un date, ou chercher une relation sérieuse.
- Ce qui est intéresant, c'est que ce sont les 2 catégories les plus représentées qui ont le taux de match moyen le plus élevé. Il y a donc une corrélation entre le fait de venir s'amuser / rencontrer de nouvelles personnes et obtenir des matchs.
- Il n'y a pas tant de différence entre les genres sur les objectifs de rendez-vous, ce qui a surement conduit aux matchs pour les mêmes raisons.

In [15]:
# We want to know what you look for in the opposite sex. 

me_vs_opposite_sex = df.groupby("gender_text")[cols_attr1].mean().round(2).reset_index()

me_vs_opposite_sex = me_vs_opposite_sex.rename(columns=({
    "attr1_1":"Attractive",
    "sinc1_1":"Sincere",
    "intel1_1":"Intelligent",
    "fun1_1":"Fun",
    "amb1_1":"Ambitious",
    "shar1_1":"Shared Interests"
}))

display(me_vs_opposite_sex)

# Renversement de la table pour préparation visualisation

me_vs_opposite_sex_melted = me_vs_opposite_sex.melt(id_vars="gender_text", var_name="Attribute", value_name="Mean Score")

# Visualiser les moyennes des hommes et femmes par attributs
fig = px.bar(me_vs_opposite_sex_melted,
             x="Attribute",
             y="Mean Score",
             color="gender_text",
             barmode="group",
             color_discrete_map={
                 "Femme":"#E75480",
                 "Homme":"#1f77b4"},
            title= "Moyenne de l'importance des attributs que les participants accordent au sexe opposé (avant-date)",
            labels={
                "gender_text":"Genre",
                "Mean Score":"Moyenne des scores",
                "Attribute":"Attributs"
            })

fig.show()

,gender_text,Attractive,Sincere,Intelligent,Fun,Ambitious,Shared Interests
0,Femme,18.04,18.28,20.95,17.28,12.92,12.56
1,Homme,26.80,16.51,19.68,17.60,8.64,11.02


# 3. Distribution des matchs par participant

In [16]:
participants = df.groupby("iid").agg(
    matches=("match","sum")
).reset_index()

participants.describe()

# visualisation

fig = px.histogram(
    participants,
    x="matches",
    nbins=20,
    title="Distribution du nombre de matchs par participant"
)
fig.update_layout(bargap=0.2)

fig.show()

In [17]:
# Affichage des matchs par participants, avec moyenne des notes reçus par attributs

participants= df.groupby("iid").agg(
    match=("match","sum"),
    attr=("attr_o","mean"),
    fun=("fun_o","mean"),
    intelligence=("intel_o",'mean'),
    sincerity=("sinc_o","mean"),
    shared=("shar_o","mean")
).round(2).reset_index().sort_values(by='match', ascending=False)

display(participants)

,iid,match,attr,fun,intelligence,sincerity,shared
511,524,14,7.00,7.71,7.86,7.75,6.05
202,208,11,7.70,7.85,7.35,7.65,6.05
106,107,11,6.89,8.11,8.00,8.06,7.18
356,366,10,7.53,7.42,8.42,8.26,7.33
18,19,9,7.70,8.50,7.70,6.80,7.70
...,...,...,...,...,...,...,...
449,461,0,4.50,5.67,7.00,6.83,4.00
53,54,0,4.42,4.84,6.75,6.32,5.00
447,459,0,7.17,6.60,7.00,6.80,5.50
445,457,0,5.67,7.20,6.80,7.67,5.60


In [18]:
model = smf.ols(
    "match ~ attr + fun + intelligence + sincerity + shared",
    data=participants
).fit()

print(model.summary())

                            OLS Regression Results                            
Dep. Variable:                  match   R-squared:                       0.159
Model:                            OLS   Adj. R-squared:                  0.152
Method:                 Least Squares   F-statistic:                     20.25
Date:                Mon, 09 Mar 2026   Prob (F-statistic):           1.60e-18
Time:                        09:54:35   Log-Likelihood:                -1147.9
No. Observations:                 540   AIC:                             2308.
Df Residuals:                     534   BIC:                             2334.
Df Model:                           5                                         
Covariance Type:            nonrobust                                         
                   coef    std err          t      P>|t|      [0.025      0.975]
--------------------------------------------------------------------------------
Intercept       -2.1806      1.012     -2.154   

- Attractivité : coefficient interessant, corrélation, et p-value significative
- Fun : coeff plus faible mais p-value significative, laisse plus place au hasard et au jugement indivividuel
- Intelligence : coeff faible et p-value non significative
- Sincérité : coeff négatif et p-value non significative
- Intérêts communs : coeff intéressant, et p-value significative

In [19]:
# Nbr de match par vague

match_ratio_wave = df.groupby("wave").agg(
    total_lignes=("match","count"),
    nbr_match_ind=("match","sum")
).reset_index()

# Nbr de rencontre physique uniques
match_ratio_wave["nbr_rdv_uniques"] = (match_ratio_wave["total_lignes"] / 2).astype(int)
match_ratio_wave["nbr_couples_matches"] = (match_ratio_wave["nbr_match_ind"] / 2).astype(int)

match_ratio_wave["match_ratio_pct"] = ((match_ratio_wave["nbr_couples_matches"] / match_ratio_wave["nbr_rdv_uniques"]) * 100).round(2)

match_ratio_wave = match_ratio_wave[['wave', 'nbr_rdv_uniques', 'nbr_couples_matches', 'match_ratio_pct']]

display(match_ratio_wave)

# Calcul du taux de match sur toute la période de l'étude
average_match_study = ((match_ratio_wave["nbr_couples_matches"].sum() / match_ratio_wave['nbr_rdv_uniques'].sum())*100).round(2)

# Visualisation

fig = px.bar(match_ratio_wave,
             x="wave",
             y="match_ratio_pct",
             title="% de match par vague",
             labels={
                 "match_ratio_pct":"% de match",
                 "wave":"vague"
             })

fig.show()

print(f"Pendant toute la période de l'étude, il y a eu :{match_ratio_wave["nbr_couples_matches"].sum()} matchs, sur {match_ratio_wave["nbr_rdv_uniques"].sum()} rendez-vous")
print(f"Soit, {average_match_study}% de match")

,wave,nbr_rdv_uniques,nbr_couples_matches,match_ratio_pct
0,1,94,30,31.91
1,2,254,27,10.63
2,3,88,11,12.50
3,4,274,55,20.07
4,5,87,24,27.59
5,6,19,4,21.05
6,7,209,37,17.70
7,8,85,16,18.82
8,9,345,58,16.81
9,10,68,13,19.12


Pendant toute la période de l'étude, il y a eu :619 matchs, sur 3596 rendez-vous
Soit, 17.21% de match


In [20]:
# Agrégation au niveau participant
df_part = (
    df.groupby(["iid", "wave"])
      .agg(
          nb_rdv=("match", "count"),
          nb_match=("match", "sum")
      )
      .reset_index()
)

# Moyenne de match individuelle
df_part["match_mean"] = (df_part["nb_match"] / df_part["nb_rdv"])*100

# Moyenne par vague
wave_mean = (
    df_part.groupby("wave")["match_mean"]
           .mean()
           .round(2)
           .reset_index()
           .rename(columns={"match_mean": "wave_mean"})
)

# Merge avec le dataframe participant
df_part = df_part.merge(wave_mean, on="wave")

df_part["match_mean_global"] = average_match_study

print("Tableau de l'ensemble des participants :")
display(df_part)

# Participants au dessus des moyennes de leur vague
df_part["above_wave_mean"] = df_part["match_mean"] > df_part["wave_mean"]
print("Tableau des participants avec un taux de match au dessus de la moyenne de leur vague :")
display(df_part[df_part["match_mean"] > df_part["wave_mean"]])

# Participants au dessus de la moyenne générale de l'étude
df_part["above_global_mean"] = df_part["match_mean"] > average_match_study
print("Tableau des participants avec un taux de match au dessus de la moyenne générale de l'étude :")
display(df_part[df_part["match_mean"] > df_part["match_mean_global"]])

# supérieur aux 2 moyennes:
participants_above_both = df_part[
    (df_part["match_mean"] > df_part["wave_mean"]) &
    (df_part["match_mean"] > average_match_study)
]

# Voir combien ils sont :
print("Au-dessus de leur vague :", df_part["above_wave_mean"].sum())
print("Au-dessus de la moyenne globale :", df_part["above_global_mean"].sum())
print("Au-dessus des deux :", len(participants_above_both))

Tableau de l'ensemble des participants :


,iid,wave,nb_rdv,nb_match,match_mean,wave_mean,match_mean_global
0,1,1,10,4,40.000000,32.99,17.21
1,2,1,10,2,20.000000,32.99,17.21
2,3,1,10,0,0.000000,32.99,17.21
3,4,1,10,2,20.000000,32.99,17.21
4,5,1,10,2,20.000000,32.99,17.21
...,...,...,...,...,...,...,...
535,548,21,21,5,23.809524,16.51,17.21
536,549,21,22,5,22.727273,16.51,17.21
537,550,21,22,4,18.181818,16.51,17.21
538,551,21,20,2,10.000000,16.51,17.21


Tableau des participants avec un taux de match au dessus de la moyenne de leur vague :


,iid,wave,nb_rdv,nb_match,match_mean,wave_mean,match_mean_global,above_wave_mean
0,1,1,10,4,40.000000,32.99,17.21,True
7,8,1,10,8,80.000000,32.99,17.21,True
8,9,1,10,7,70.000000,32.99,17.21,True
9,10,1,2,1,50.000000,32.99,17.21,True
12,13,1,10,4,40.000000,32.99,17.21,True
...,...,...,...,...,...,...,...,...
533,546,21,21,6,28.571429,16.51,17.21,True
535,548,21,21,5,23.809524,16.51,17.21,True
536,549,21,22,5,22.727273,16.51,17.21,True
537,550,21,22,4,18.181818,16.51,17.21,True


Tableau des participants avec un taux de match au dessus de la moyenne générale de l'étude :


,iid,wave,nb_rdv,nb_match,match_mean,wave_mean,match_mean_global,above_wave_mean,above_global_mean
0,1,1,10,4,40.000000,32.99,17.21,True,True
1,2,1,10,2,20.000000,32.99,17.21,False,True
3,4,1,10,2,20.000000,32.99,17.21,False,True
4,5,1,10,2,20.000000,32.99,17.21,False,True
5,6,1,10,2,20.000000,32.99,17.21,False,True
...,...,...,...,...,...,...,...,...,...
533,546,21,21,6,28.571429,16.51,17.21,True,True
535,548,21,21,5,23.809524,16.51,17.21,True,True
536,549,21,22,5,22.727273,16.51,17.21,True,True
537,550,21,22,4,18.181818,16.51,17.21,True,True


Au-dessus de leur vague : 230
Au-dessus de la moyenne globale : 234
Au-dessus des deux : 207


In [21]:
# Colonnes attributs reçus
attr_cols = ["attr_o","sinc_o","intel_o","fun_o","amb_o","shar_o"]

# Moyenne des attributs reçus par participant
df_attr_mean = (
    df.groupby("iid")[attr_cols]
      .mean()
      .reset_index()
)

# Colonnes profil individuelles 
cols_profil_base = [
    "iid",
    "gender_text",
    "age",
    "field",
    "race_text",
    "goal_text",
    "career",
    "imprace",
    "imprelig",
    "exphappy",
    "expnum"
]

# Supprimer les doublons pour être au niveau individu
df_profil_base = df[cols_profil_base].drop_duplicates(subset="iid")

# Fusion profil + moyennes reçues
df_profil = df_profil_base.merge(df_attr_mean, on="iid", how="left")

# Fusion avec stats match
df_part_full = df_part.merge(df_profil, on="iid", how="left")

# Participants sans matchs
df_no_match = df_part_full[df_part_full["nb_match"] == 0]

# Variable pour la table de profil 
cols_quant = [
    "age",
    "imprace",
    "imprelig",
    "exphappy",
    "expnum",
    "attr_o","sinc_o","intel_o","fun_o","amb_o","shar_o"
]

cols_cat = ["field", "race_text", "goal_text", "career"]

# Fonctions pour créer les 2 tables de profils types
def top_category_with_pct(x):
    counts = x.value_counts(normalize=True) * 100
    top_cat = counts.idxmax()
    top_pct = counts.max()
    return f"{top_cat} ({top_pct:.1f}%)"


def build_profile_table(df_input):

    # Moyennes numériques
    df_mean = (
        df_input
        .groupby("gender_text")[cols_quant]
        .mean()
        .round(2)
    )

    # Catégories dominantes
    df_cat = (
        df_input
        .groupby("gender_text")[cols_cat]
        .agg(lambda x: top_category_with_pct(x))
    )

    return pd.concat([df_mean, df_cat], axis=1)

In [22]:
# Top performers en matchs

df_top_full = df_part_full[
    df_part_full["iid"].isin(participants_above_both["iid"])
]

df_profil_most_match = build_profile_table(df_top_full)

print("----- Profil type des participants avec le plus de matchs -----")
display(df_profil_most_match)

----- Profil type des participants avec le plus de matchs -----


,age,imprace,imprelig,exphappy,expnum,attr_o,sinc_o,intel_o,fun_o,amb_o,shar_o,field,race_text,goal_text,career
gender_text,,,,,,,,,,,,,,,
Femme,25.62,3.91,4.08,5.26,8.38,6.91,7.42,7.43,6.81,6.78,5.90,Law (5.1%),European/Caucasian-American (49.0%),Seemed like a fun night out (48.0%),professor (4.1%)
Homme,26.44,2.97,2.75,6.02,6.92,6.49,7.34,7.68,6.84,7.12,5.94,Business (12.3%),European/Caucasian-American (67.0%),Seemed like a fun night out (38.7%),Finance (4.7%)


In [23]:
# Participants sans matchs

df_profil_no_match = build_profile_table(df_no_match)

print("----- Profil type des participants sans match -----")
display(df_profil_no_match)

----- Profil type des participants sans match -----


,age,imprace,imprelig,exphappy,expnum,attr_o,sinc_o,intel_o,fun_o,amb_o,shar_o,field,race_text,goal_text,career
gender_text,,,,,,,,,,,,,,,
Femme,26.24,4.83,4.64,5.00,4.45,5.98,7.19,7.16,6.28,6.35,5.23,Social Work (11.9%),European/Caucasian-American (54.2%),To meet new people (37.3%),Social Worker (5.1%)
Homme,26.12,3.48,2.54,5.64,4.28,5.07,7.22,7.45,5.69,6.96,4.72,MBA (14.0%),European/Caucasian-American (44.0%),Seemed like a fun night out (38.0%),Finance (6.0%)


In [24]:
# ---- Participants qui n'ont eu aucun match -----

print("----- Participants qui n'ont eu aucun match : -----")
print(f"Nombre de participants sans match : {len(df_no_match)}")
print(f"Soit : {((len(df_no_match) / total_men_women_by_waves['count'].sum()) *100).round(2)}%")


----- Participants qui n'ont eu aucun match : -----
Nombre de participants sans match : 110
Soit : 20.37%


In [25]:
cols_radar = ["attr_o","sinc_o","intel_o","fun_o","amb_o","shar_o","imprace","imprelig","exphappy"]

# Moyennes Top performers
top_means = (
    df_top_full
    .groupby("gender_text")[cols_radar]
    .mean()
    .reset_index()
)

# Moyennes No match
no_match_means = (
    df_no_match
    .groupby("gender_text")[cols_radar]
    .mean()
    .reset_index()
)

def plot_radar_plotly(gender):

    # Filtrer par genre
    df_top_gender = top_means[top_means["gender_text"] == gender]
    df_no_gender = no_match_means[no_match_means["gender_text"] == gender]

    # Ajouter colonne groupe
    df_top_gender = df_top_gender.assign(group="Top performers")
    df_no_gender = df_no_gender.assign(group="No match")

    # Concaténer
    df_plot = pd.concat([df_top_gender, df_no_gender])

    # Format long
    df_plot = df_plot.melt(
        id_vars=["gender_text", "group"],
        value_vars=cols_radar,
        var_name="variable",
        value_name="value"
    )

    # Création radar
    fig = px.line_polar(
        df_plot,
        r="value",
        theta="variable",
        color="group",
        markers=True,
        hover_data={
            "value":":.2f",
            "variable":True,
            "group":True
        },
        line_close=True,
        title=f"Radar comparatif - {gender}"
    )

    fig.update_traces(fill="toself")
    fig.show()

# Radar Hommes
plot_radar_plotly("Homme")

# Radar Femmes
plot_radar_plotly("Femme")

**INTERPRETATION**
Cette visualisation nous aide beaucoup à comprendre ce qui fait que certains ont plus de matchs que d'autres :
1. Les variables discrimantes comme : l'importance d'être de la même origine ethnique et de la même religion, font que certaines personnes sont plus sélectives et obtiennent donc moins de match. C'est notament remarquable chez les femmes, plus que chez les hommes.
2. On constate que la note moyenne d'attractivité compte beaucoup pour les 2 genres, avec une plus grande différence chez les hommes.
3. Chez les hommes, la partage d'intérêts communs et le fait qu'ils soient amusant, sont des critères de séduction visiblement fort pour obtenir plus de matchs.

# 4. L'attractivité influence-t-elle les matchs ?

In [26]:
# Relation entre l'attractivité et le nombre de match
fig = px.scatter(
    participants,
    x="attr",
    y="match",
    trendline="ols",
    title="Relation entre l'attirance et le nombre de matchs"
)

fig.show()

On constate une corrélation avec la note reçue d'attractivité et le nombre de matchs.

In [27]:
fig = px.box(
    df,
    x="match",
    y="attr",
    title="Attractivité perçue selon qu'il y ait match ou non",
    labels={"match":"Match","attr":"Attractivité"}
)

fig.show()

# 5. Quels facteurs influencent la décision YES ?

In [29]:
model = smf.logit(
    "dec ~ attr + fun + intel + sinc + shar + int_corr",
    data=df
).fit()

print(model.summary())

Optimization terminated successfully.
         Current function value: 0.511645
         Iterations 6
                           Logit Regression Results                           
Dep. Variable:                    dec   No. Observations:                 7071
Model:                          Logit   Df Residuals:                     7064
Method:                           MLE   Df Model:                            6
Date:                Mon, 09 Mar 2026   Pseudo R-squ.:                  0.2520
Time:                        09:54:36   Log-Likelihood:                -3617.8
converged:                       True   LL-Null:                       -4836.8
Covariance Type:            nonrobust   LLR p-value:                     0.000
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept     -5.4351      0.187    -29.035      0.000      -5.802      -5.068
attr           0.5394      0.

In [30]:
# Visualisation des coeff

coef = model.params.drop("Intercept")

coef_df = pd.DataFrame({
    "variable":coef.index,
    "coef":coef.values
})

fig = px.bar(
    coef_df,
    x="variable",
    y="coef",
    title="Effet des caractéristiques sur la probabilité de dire YES"
)

fig.show()

Pour identifier les facteurs qui influencent la décision de dire “yes”, nous utilisons une régression logistique.

Ce modèle estime la probabilité de vouloir revoir un partenaire en fonction de ses caractéristiques.

Les résultats montrent que :
- l’attractivité est le facteur le plus important
- le caractère fun et les intérêts communs jouent aussi un rôle positif
- l’intelligence ou la sincérité ont un impact plus limité dans la première impression.

Cela suggère que l’attraction initiale repose principalement sur des signaux immédiats et perceptibles.

In [31]:
# Calcul du nombre réel de décisions postives reçues

real_success = (
    df.groupby("iid")["dec_o"]
      .sum()
      .reset_index()
      .rename(columns={"dec_o":"real_yes"})
)

# Fusion avec la confiance initiale

confidence = df[["iid","gender_text","expnum"]].drop_duplicates()

calibration = confidence.merge(real_success, on="iid")

# Groupes forte/faible confiance

median_conf = calibration["expnum"].median()

calibration["confidence_group"] = calibration["expnum"].apply(
    lambda x: "Haute confiance" if x >= median_conf else "Basse confiance"
)

display(calibration)

# Visualisation boxplot pour dispersion des données

fig = px.box(
    calibration,
    x="confidence_group",
    y="real_yes",
    color="confidence_group",
    title="Décisions positives reçues par niveau de confiance en soi",
    labels={
        "confidence_group":"Groupe de confiance",
        "real_yes":"Succès réel"
        }
)

fig.show()

# Visualisation scatterplot

fig = px.scatter(
    calibration,
    x="expnum",
    y="real_yes",
    color="gender_text",
    trendline="ols",
    title="Corrélation entre la confiance en soi et le succès réel <br><sup>Sur 20 personnes, combien attendez-vous qu'elles soient intéressées par vous ?</sup>",
    labels={
        "expnum":"Note de confiance",
        "real_yes":"Succès réel",
        "gender_text":"Genre"
    },
    color_discrete_map={
                 "Femme":"#E75480",
                 "Homme":"#1f77b4"}
)
fig.show()

,iid,gender_text,expnum,real_yes,confidence_group
0,1,Femme,2.0,5,Basse confiance
1,2,Femme,5.0,6,Haute confiance
2,3,Femme,2.0,5,Basse confiance
3,4,Femme,2.0,6,Basse confiance
4,5,Femme,10.0,3,Haute confiance
...,...,...,...,...,...
535,548,Homme,NaN,10,Basse confiance
536,549,Homme,NaN,10,Basse confiance
537,550,Homme,NaN,6,Basse confiance
538,551,Homme,NaN,10,Basse confiance


In [32]:
# Est-ce que l'âge est un facteur d'influence sur le nombre de décisions positives reçues ?

decision_stats = (
    df.groupby(["age", "gender_text"])
      .agg(
          total_dates=("dec_o", "count"),
          positive_decisions=("dec_o", "sum"),
          positive_ratio=("dec_o", "mean")
      )
      .reset_index()
)

# Visualisation line chart
fig = px.line(
    decision_stats,
    x="age",
    y="positive_ratio",
    color="gender_text",
    markers=True,
    title="Représentation du taux de décisions positives reçues, par âge, et par sexe",
    labels={
        "gender_text":"Genre",
        "age":"Age des participants",
        "positive_ratio":"Ratio de décisions positives"
    },
    color_discrete_map={
                 "Femme":"#E75480",
                 "Homme":"#1f77b4"}
)

fig.show()

# 6. Similarité entre partenaires

In [33]:
table_gender_race_decision_ratio = df.groupby(["gender_text","race_text"])["dec_o"].mean().round(2).reset_index(name="positive_decision_ratio")

display(table_gender_race_decision_ratio)

fig = px.bar(table_gender_race_decision_ratio, 
             x="race_text",
             y="positive_decision_ratio",
             color="gender_text",
             barmode="group",
             color_discrete_map={
                 "Femme":"#E75480",
                 "Homme":"#1f77b4"},
            title="Ratio de décisions positives reçues,le soir de l'événement, par genre<br> et par ethnie d'origine",
            labels={
                "gender_text":"Genre",
                "race_text":"Ethnie d'origine",
                "positive_decision_ratio":"Ratio de décision positive"
            })
fig.show()

,gender_text,race_text,positive_decision_ratio
0,Femme,Asian/Pacific Islander/Asian-American,0.42
1,Femme,Black/African American,0.48
2,Femme,European/Caucasian-American,0.51
3,Femme,Latino/Hispanic American,0.56
4,Femme,Other,0.46
5,Homme,Asian/Pacific Islander/Asian-American,0.28
6,Homme,Black/African American,0.35
7,Homme,European/Caucasian-American,0.42
8,Homme,Latino/Hispanic American,0.40
9,Homme,Other,0.33


In [34]:
# Visualisation sous forme de carte de chaleur, avec le ratio de décision positive par genre et par ethnie d'origine

pivot = table_gender_race_decision_ratio.pivot(
    index="gender_text",
    columns="race_text",
    values="positive_decision_ratio"
)

fig = px.imshow(
    pivot,
    aspect="auto",
    text_auto=True,
    color_continuous_scale="Blues",
    title="Carte des chaleurs des ratio de décisions positives par genre et ethnie",
    labels=dict(
        x="Genre",
        y="Ethnie d'origine"
    )
)

fig.show()

**INTERPRETATION :**

Grâce à la carte des température, on peut facilement constater que les participants qui ont le plus de succès sont les femmes d'origine Latino/Hispanic, à l'inverse, ceux qui ont le moins de succès sont les hommes d'origine asiatique.

In [35]:
# Nombre de personnes uniques pour chaque niveau d'importance de la même origine ethnique : 0 = pas important, 10 = très important

importance_same_race = df.groupby(["imprace","gender_text"])["iid"].nunique().reset_index(name="n_persons")

importance_same_race["ratio"] = round(importance_same_race["n_persons"] / importance_same_race["n_persons"].sum(),2)

# Afficher sous forme de tablau
display(importance_same_race)

# Visualisation graphique
fig = px.bar(importance_same_race, 
             x="imprace",
             y="n_persons",
             title="Nombre de participants par niveau d'importance<br> de la même origine ethnique",
             color="gender_text",
             barmode="group",
             color_discrete_map={
                 "Femme":"#E75480",
                 "Homme":"#1f77b4"},
            labels={
                "imprace":"Importance de la même origine (note sur 10)",
                "n_persons":"Nombre de participants",
                "gender_text":"Genre"}
             )
fig.update_xaxes(tickmode='linear')

fig.show()

,imprace,gender_text,n_persons,ratio
0,0.0,Femme,1,0.00
1,1.0,Femme,78,0.15
2,1.0,Homme,106,0.20
3,2.0,Femme,33,0.06
4,2.0,Homme,26,0.05
5,3.0,Femme,30,0.06
6,3.0,Homme,35,0.07
7,4.0,Femme,16,0.03
8,4.0,Homme,17,0.03
9,5.0,Femme,24,0.05


In [36]:
# ceux qui disent que l'origine ethnique est très importante ont ils plus tendance à accepter uniquement les partenaires de même origine ?

analysis = df.groupby(["imprace", "samerace"])["dec"].mean().reset_index(name="yes_rate")

table = (
    analysis.pivot(
        index="imprace",
        columns="samerace",
        values="yes_rate"
    )
    .rename(columns={0: "Different_race", 1: "Same_race"})
    .reset_index()
)

# Visualisation de l'écart entre les attentes de la même origine ethnie, et la réalité de la décision

px.line(
    table,
    x="imprace",
    y=["Same_race", "Different_race"],
    markers=True,
    title="Comparaison entre l'attente de la même origine ethnique, et réalité de la décision<br><sup>Same_race : proportion réelle de décisions positives quand le partenaire est de la même origine<br>Different_race : proportion réelle de décisions positives quand le partenaire est d’une origine différente</sup>",
    labels={
        "imprace":"Importance d'être de la même origine",
        "value":"Ratio de réalité de la décision"
    }
)

**INTERPRETATION :**

- Pour ceux qui apportent peu d'importance à la même origine ethnique, les deux courbes sont proches, ils sont donc cohérent avec leur discours. 
- Ceux qui sont dans une importance moyenne, plus quelqu'un dit que l'origine ethnique est important, moins il accepte les partenaires d'une autre origine. 
- Ceux pour qui l'importance est forte, l'effet est plus marqué, on parle alors d'homogamie ethnique comportementale.

In [37]:
table = pd.crosstab(df["match"], df["samerace"])

chi2_contingency(table)

Chi2ContingencyResult(statistic=np.float64(0.979496122236936), pvalue=np.float64(0.32232323712806465), dof=1, expected_freq=array([[3614.98334721, 2346.01665279],
       [ 755.01665279,  489.98334721]]))

H0 :le fait d’être de la même origine n’influence pas les matchs

H1 :être de la même origine influence les matchs

Chi2 : 0.979

P-value = 0.322

Pas de relation significative statistiquement entre “être de la même origine”, et la probabilité d’un match

In [38]:
fig = px.histogram(
    df,
    x="samerace",
    color="match",
    barmode="group",
    title="Effet du fait d'être de la même origine sur les matchs"
)

fig.show()

Nous avons également testé si les personnes sont plus susceptibles de matcher avec des partenaires similaires.

Par exemple, nous examinons si le fait d’être de la même origine ethnique influence les matchs.

Pour cela, nous utilisons un test du chi-carré.

Le résultat montre une p-value supérieure à 0.05, ce qui signifie que l’effet n’est pas statistiquement significatif.

En revanche, les intérêts communs semblent avoir un impact plus positif sur l’attraction.

In [39]:
# Match par participants

participants = df.groupby("iid").agg(
matches=("match","sum")
).reset_index()

# Visualisation
fig = px.histogram(
participants,
x="matches",
nbins=20,
title="Distribution of Matches per Participant"
)
fig.update_layout(bargap=0.2)

fig.show()

### Est-ce que les premiers rendez-vous reçoivent plus de réponses positives de match ?

In [40]:
# order : the number of date that night when met partner
# dec_o : decision of partner the night of event

order_yes = df[df["dec_o"] == 1].groupby(["order","gender_text"])["iid"].count().reset_index(name="n_yes")
order_total = df.groupby("order")["iid"].count().reset_index(name="n_total")
order_yes = order_yes.merge(order_total, on="order")
order_yes["yes_rate"] = round(order_yes["n_yes"] / order_yes["n_total"],2)

display(order_yes)

# Visualisation:
fig = px.bar(order_yes, 
             x="order", 
             y="yes_rate", 
             color='gender_text',
             barmode="group",
             title="Taux de OUI en fonction de l'ordre du date dans la soirée",
             color_discrete_map={
                 "Femme":"#E75480",
                 "Homme":"#1f77b4"},
            labels={
                    "gender_text":"Genre",
                    "order":"Ordre dans la soirée",
                    "yes_rate":"Taux de OUI"
            })
fig.show()

,order,gender_text,n_yes,n_total,yes_rate
0,1,Femme,128,480,0.27
1,1,Homme,112,480,0.23
2,2,Femme,103,473,0.22
3,2,Homme,86,473,0.18
4,3,Femme,107,488,0.22
5,3,Homme,96,488,0.20
6,4,Femme,120,494,0.24
7,4,Homme,105,494,0.21
8,5,Femme,122,486,0.25
9,5,Homme,92,486,0.19


In [41]:
# Taux de réponses positives dans les derniers passages est calculé sur beaucoup moins de personnes que le premier. Consolidation avec un test statistique :

corr, p_value = stats.pearsonr(df.dropna(subset=['order', 'dec_o'])['order'], 
                               df.dropna(subset=['order', 'dec_o'])['dec_o'])
print(f"Corrélation : {corr:.4f}, p-value : {p_value:.4f}")


Corrélation : -0.0138, p-value : 0.2411


**INTERPRETATION :**

- la corrélation négative est très faible (r = -0.03). Cela signifie que l'ordre du passage n'explique qu'une fraction infime de la décision finale.
- la pvalue < 0.05 indique bien que notre résultat est significatif et pas du au hasard.

L'ordre ne pénalise par injustement les participants.

# CONCLUSION 

Cette analyse met en évidence trois résultats principaux :

1️⃣ L’attractivité physique est le facteur le plus déterminant dans l’attraction initiale.

2️⃣ Les traits de personnalité comme être fun ou partager des intérêts communs augmentent également la probabilité d’un match.

3️⃣ Les caractéristiques démographiques comme l’origine ethnique ne semblent pas influencer significativement les décisions.

Ces résultats suggèrent que dans un contexte de rencontre rapide, les décisions reposent principalement sur des signaux visibles et immédiats, plutôt que sur des facteurs plus profonds.


### 1. Le paradoxe "Dire vs Faire"
L'enseignement majeur de cette étude est l'écart entre les préférences déclarées et les choix réels.

- Les Hommes : Déclarent chercher de l'intelligence et de la sincérité, mais leurs décisions finales sont massivement corrélées à l'attractivité physique de la partenaire.
- Les Femmes : Affichent une plus grande exigence sur l'ensemble des critères. Si l'attractivité reste primordiale, l'intelligence et l'ambition du partenaire pèsent réellement plus lourd dans leur décision finale que chez les hommes.

### 2. Le poids de la perception de soi

L'analyse du "Gap de perception" montre que la lucidité est une arme de séduction :
- Les participants ayant une auto-évaluation proche de la note reçue par leurs partenaires tendent à obtenir plus de matchs.
- Une surconfiance marquée (se noter 9/10 quand on reçoit une moyenne de 5/10) est souvent corrélée à un taux de rejet plus élevé, suggérant que l'adéquation entre l'image de soi et la réalité perçue par l'autre joue un rôle social subtil.

### 3. L'influence négligeable du timing et des intérêts communs

- La fatigue décisionnelle : Bien que statistiquement significative (p < 0.05), la baisse du taux de "Oui" au fil de la soirée est négligeable (r = -0.03). Le format de 4 minutes semble donc efficace pour maintenir l'intérêt.
- Les intérêts communs : Étonnamment, la corrélation des attributs (int_corr) n'est pas un prédicteur fort du match. On "match" sur une dynamique interpersonnelle et physique plutôt que sur une liste d'attributs partagés.

### 💡 Recommandations pour Tinder

Pour optimiser l'engagement des utilisateurs, la plateforme devrait :
- Valoriser les profils complets : Puisque les femmes accordent une importance réelle à l'intelligence et l'ambition, l'algorithme devrait mettre en avant ces informations (études, carrière) pour le public féminin.
- Encourager la lucidité : Des systèmes de "feedback" anonymes (comme les attributs les plus cités par les partenaires) pourraient aider les utilisateurs à mieux comprendre leur image et à ajuster leurs attentes ou leur présentation.
- Ne pas sur-pondérer les attributs : L'algorithme de matching gagne à privilégier les traits de personnalité et l'attrait visuel plutôt que la simple accumulation d'attributs communs.

Réalisé par : Samuel PLASSMANN

Date : 09/03/2026